# Perfilado de la Carga y las Interrupciones de la Red Regional con PROC CHART

## Resumen ejecutivo

Un analista de operaciones de red usa PROC CHART para perfilar una muestra sintética de lecturas de medidores de circuitos alimentadores en cuatro regiones de servicio y cuatro fuentes de generación. El cuaderno recorre gráficos de barras verticales, barras horizontales, bloques y sectores para resumir la combinación de generación, comparar la carga media y total por región, exponer el pico de demanda vespertino por hora, y clasificar los minutos de interrupción por fuente —el tipo de exploración rápida de salida de texto que un equipo de energía y servicios públicos ejecuta antes de comprometerse con un informe gráfico. El paso DATA solicita 1,200 filas; este cuaderno se generó en modo sin licencia, que limita el conjunto de datos de trabajo a las primeras 100 lecturas, por lo que cada gráfico a continuación resume esa muestra de 100 filas.

## Fuentes de datos

| Conjunto de datos | Filas | Descripción |
|---------|------|-------------|
| `grid_ops` | 100 (muestra sintética) | Una fila por lectura de medidor de circuito alimentador en una red regional, generada en línea con `call streaminit(20260531)` y `rand()`. El bucle del paso DATA pide 1,200 lecturas, pero el modo sin licencia limita el conjunto de datos guardado a 100 observaciones, por lo que los gráficos a continuación describen esas 100 filas. |

# Perfilado de la carga y las interrupciones de la red regional con PROC CHART

PROC CHART produce gráficos de barras, bloques y sectores basados en caracteres directamente en el listado —sin necesidad de un dispositivo ODS Graphics. Eso lo convierte en una herramienta rápida de exploración de primera pasada para un equipo de operaciones de red que quiere *ver* la forma de sus datos de carga y fiabilidad antes de construir visualizaciones pulidas con GCHART o SGPLOT.

En este cuaderno:

1. Generamos un mes sintético de lecturas de medidores de circuitos alimentadores para un operador de red regional.
2. Graficamos la **combinación de generación** (proporción de lecturas por fuente).
3. Comparamos la **carga media y total entregada** entre regiones de servicio.
4. Exponemos el **pico de demanda vespertino** por hora del día.
5. Usamos un **gráfico de bloques** para cruzar región contra fuente de generación.
6. Clasificamos los **minutos de interrupción** por fuente para encontrar el suministro menos fiable.

Cada sentencia y opción a continuación es sintaxis estándar de PROC CHART de SAS 9.4.

## Paso 1 — Generar los datos sintéticos de operaciones

El paso DATA a continuación fabrica lecturas de medidores en un bucle de 1,200 iteraciones. A cada lectura se le asigna una región de servicio, una fuente de generación (predomina Gas, con Wind, Solar e Hydro completando el resto) y una hora del día. La carga es mayor en la ventana vespertina de 17:00–21:00 (y marcamos esas lecturas como pico), y extraemos los minutos de interrupción de una distribución de Poisson. `call streaminit` fija la semilla aleatoria para que los datos sean reproducibles.

La NOTA del registro muestra el resultado práctico: esta ejecución está en modo sin licencia, por lo que el conjunto de datos `grid_ops` guardado se limita a las primeras 100 de esas lecturas (100 filas, 7 columnas). Cada gráfico que sigue resume esa muestra de 100 filas, y cada registro de PROC CHART confirma que leyó 100 filas.

In [1]:
/* Synthetic feeder-circuit operations for a regional grid operator */
DATOS grid_ops;
    LLAMAR streaminit(20260531);
    LONGITUD region $12 source $9 peak_flag $3;
    ARREGLO regions[4] $12 _temporary_
        ('North','South','East','West');
    HACER meter_id = 1 HASTA 1200;
        r = ceil(4 * rand('uniform'));
        region = regions[r];
        u = rand('uniform');
        SI u < 0.42 ENTONCES source = 'Gas';
        SINO SI u < 0.70 ENTONCES source = 'Wind';
        SINO SI u < 0.88 ENTONCES source = 'Solar';
        SINO source = 'Hydro';
        hour = floor(24 * rand('uniform'));
        BASE = 18 + 5 * (region = 'North') + 3 * (region = 'West');
        SI hour >= 17 AND hour <= 21 ENTONCES HACER;
            load_mwh = BASE + 12 + 6 * rand('normal');
            peak_flag = 'Yes';
        END;
        SINO HACER;
            load_mwh = BASE + 4 * rand('normal');
            peak_flag = 'No';
        END;
        SI load_mwh < 0 ENTONCES load_mwh = 0;
        outage_min = rand('poisson', 2.5);
        SALIDA;
    END;
    ELIMINAR r u BASE;
EJECUTAR;

NOTE: DATA grid_ops

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote grid_ops (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds


## Paso 2 — Combinación de generación

¿Qué proporción de lecturas representa cada fuente de generación? Un gráfico de barras verticales con `TYPE=PERCENT` responde esto directamente: las alturas de las barras son el porcentaje de todas las observaciones que caen en cada categoría de fuente. Como `source` es una variable de carácter, PROC CHART trata sus valores como categorías discretas automáticamente.

In [2]:
PROCEDIMIENTO chart DATOS=grid_ops;
    VBAR source / type=PERCENT;
EJECUTAR;


Percent of SOURCE

         |              *****       
         |              *****       
   40.00 +              *****       
         |              *****       
         |              *****       
   30.00 +              *****       
         |       *****  *****       
         |       *****  *****       
   20.00 +       *****  *****       
         |       *****  *****       
         |*****  *****  *****       
         |*****  *****  *****  *****
   10.00 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |                          
         +--------------------------+
          Hydro  Wind    Gas   Solar
                    SOURCE


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Paso 3 — Carga media entregada por región

Ahora pasamos de contar a resumir una medición. Nombrar `load_mwh` en `SUMVAR=` con `TYPE=MEAN` hace que la longitud de la barra sea la carga media por región, y solicitamos las columnas de estadísticas explícitamente: `MEAN` imprime el promedio junto a cada barra y `CFREQ` añade una columna de frecuencia acumulada. North registra la mayor carga media entregada (media de aproximadamente 28.0 MWh), consistente con el desplazamiento regional incorporado en los datos, mientras que South es la más baja (aproximadamente 19.8 MWh).

También pasamos `ASCENDING`, con la intención de ordenar las barras de la media más baja a la más alta. Observe en la salida que las barras **no** se reordenan —aparecen en orden de categoría (West, South, East, North), con medias 24.2, 19.8, 21.7, 28.0 que no van de la más corta a la más larga. El reordenamiento de barras por la estadística del gráfico aún no está implementado en esta implementación de PROC CHART, por lo que `ASCENDING`/`DESCENDING` se aceptan pero actualmente no tienen efecto; lea la clasificación en la columna `Mean` impresa en su lugar.

In [3]:
PROCEDIMIENTO chart DATOS=grid_ops;
    HBAR region / SUMVAR=load_mwh type=mean
                  cfreq mean ascending;
EJECUTAR;


Mean of REGION

REGION                                                  Mean           N     Percent
                                                                                    
  West  **********************************             24.17       24.17       23.00
 South  ****************************                   19.80       43.97       21.00
  East  *******************************                21.72       65.69       29.00
 North  ****************************************       28.03       93.72       27.00


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Paso 4 — Carga total por región

Aquí `TYPE=SUM` hace que cada barra sea la carga *total* entregada de la región en lugar del promedio, por lo que las barras más altas marcan las regiones que entregan más energía agregada a través de las lecturas muestreadas. North vuelve a liderar la carga total entregada.

La sentencia también solicita `SUBGROUP=peak_flag`, pidiendo a PROC CHART que divida cada barra según si la lectura cayó en la ventana del pico vespertino. En SAS esto segmenta cada barra con un glifo distinto por nivel de subgrupo e imprime una leyenda. Esta implementación aún no representa la segmentación por subgrupos (una brecha de capacidad documentada), por lo que las barras salen como series sólidas de `*` sin desglose por segmento —los *totales* de las barras son correctos, pero la división pico-vs-fuera-de-pico no se muestra aquí. Para ver cuánta carga cae en las horas pico, use la vista por hora del día en el Paso 5.

In [4]:
PROCEDIMIENTO chart DATOS=grid_ops;
    VBAR region / SUMVAR=load_mwh type=sum
                  SUBGROUP=peak_flag;
EJECUTAR;


Sum of REGION

         |                     *****
         |                     *****
         |                     *****
     600 +              *****  *****
         |*****         *****  *****
         |*****         *****  *****
         |*****         *****  *****
     400 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
     200 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |                          
         +--------------------------+
          West   South  East   North
                    REGION


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Paso 5 — Forma de la carga a lo largo del día

`hour` es continua, por lo que fijamos el agrupamiento nosotros mismos con una lista explícita `MIDPOINTS=` en centros de 4 horas (2, 6, 10, 14, 18, 22). Cada barra muestra la carga media entregada para las lecturas cercanas a esa hora. La barra centrada en 18 debería destacar —ese es el pico de demanda vespertino que incorporamos en los datos.

In [5]:
PROCEDIMIENTO chart DATOS=grid_ops;
    VBAR hour / SUMVAR=load_mwh type=mean
                midpoints=2 6 10 14 18 22;
EJECUTAR;


Mean of HOUR

         |                            *****       
         |                            *****       
   30.00 +                            *****       
         |                            *****  *****
         |                            *****  *****
         |              *****         *****  *****
   20.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
   10.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |                                        
         +----------------------------------------+
            2      6     10     14     18     22  
                            HOUR


NOTE: PROC CHART data=grid_ops


## Paso 6 — Región por fuente de generación (gráfico de bloques)

Un gráfico BLOCK representa una pequeña tabla de dos vías como una cuadrícula de bloques. Con `GROUP=source` y `SUMVAR=load_mwh / TYPE=MEAN`, el gráfico cruza cada región contra la fuente de generación que la abastece, con la altura del bloque proporcional a la carga media —una forma compacta de detectar qué combinaciones de región/fuente acarrean la mayor carga media.

In [6]:
PROCEDIMIENTO chart DATOS=grid_ops;
    BLOCK region / SUMVAR=load_mwh type=mean
                   GROUP=source;
EJECUTAR;


Block chart of Mean of REGION by SOURCE

                          /####\
  /####\                  /####\
  /####\          /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  +----+  +----+  +----+  +----+
   West   South    East   North 
             REGION


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Paso 7 — Combinación de generación como gráfico de sectores

La misma información de proporción por fuente del Paso 2, dibujada como un gráfico de sectores. PIE con `TYPE=PERCENT` dimensiona cada sector por su porcentaje del total de lecturas e imprime una leyenda de los porcentajes de los sectores junto a la figura.

In [7]:
PROCEDIMIENTO chart DATOS=grid_ops;
    PIE source / type=PERCENT;
EJECUTAR;


Pie chart of SOURCE

          SOURCE     Percent   Percent  Slice
----------------  ----------  --------  ------------------------------
           Hydro       14.00     14.0%  ####
            Wind       28.00     28.0%  ********
             Gas       45.00     45.0%  ++++++++++++++
           Solar       13.00     13.0%  OOOO


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Paso 8 — Minutos de interrupción por fuente

Finalmente clasificamos la fiabilidad. `SUMVAR=outage_min` con `TYPE=SUM` totaliza los minutos de interrupción por fuente. Pasamos `DESCENDING` para intentar hacer flotar la fuente de peor desempeño hasta arriba, pero como en el Paso 3 las barras no se reordenan —se imprimen en orden de categoría (Hydro, Wind, Gas, Solar) y el reordenamiento de barras aún no está implementado. Lea la clasificación en la columna `Sum` impresa: Gas representa la mayor cantidad de minutos totales de interrupción en esta muestra (122), muy por delante de Wind (64), Solar (43) e Hydro (38). Eso concuerda con la combinación de generación —Gas abastece la mayoría de las lecturas, por lo que acumula la mayor cantidad de minutos de interrupción en total.

In [8]:
PROCEDIMIENTO chart DATOS=grid_ops;
    HBAR source / SUMVAR=outage_min type=sum DESCENDENTE;
EJECUTAR;


Sum of SOURCE

SOURCE                                                   Sum        Cum.     Percent
                                                                     Sum            
 Hydro  ************                                      38          38       14.00
  Wind  *********************                             64         102       28.00
   Gas  ****************************************         122         224       45.00
 Solar  **************                                    43         267       13.00


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Interpretación de los resultados

Leer los gráficos en conjunto da al equipo de operaciones una imagen situacional rápida (sobre la muestra de 100 filas que capturó esta ejecución):

- **Combinación de generación (Pasos 2 y 7).** Gas acarrea la mayor proporción de lecturas (aproximadamente 45%), con Wind en segundo lugar (aproximadamente 28%) e Hydro y Solar rezagados (aproximadamente 14% y 13%) —la barra vertical y el gráfico de sectores cuentan la misma historia de dos maneras, una comprobación de coherencia útil.
- **Carga por región (Pasos 3 y 4).** El HBAR de carga media muestra a North con la mayor carga media entregada (media ~28 MWh) y a South la más baja (~20 MWh), consistente con el desplazamiento regional incorporado en los datos. El gráfico SUM confirma que North también entrega la mayor energía total.
- **Forma de la carga diaria (Paso 5).** La barra de punto medio centrada en la hora 18 es claramente la más alta, confirmando el pico de demanda de 17:00–21:00 que incorporamos en los datos —exactamente donde una empresa de servicios públicos enfocaría la respuesta a la demanda y la planificación de capacidad.
- **Fiabilidad (Paso 8).** Totalizar los minutos de interrupción por fuente hace emerger a Gas como el mayor contribuyente de tiempo de inactividad en esta muestra (122 minutos), el siguiente objetivo natural para la revisión de mantenimiento —aunque eso refleja principalmente que Gas abastece la mayoría de las lecturas.

Dos opciones de visualización usadas aquí —el reordenamiento de barras `ASCENDING`/`DESCENDING` (Pasos 3 y 8) y la segmentación de barras `SUBGROUP=` (Paso 4)— son aceptadas por el analizador pero aún no representadas por esta implementación de PROC CHART, por lo que las clasificaciones y divisiones se leen en las columnas de estadísticas impresas en lugar del orden o el sombreado de las barras.

PROC CHART es solo de salida de caracteres, por lo que para visuales listos para presentar a la junta el equipo trasladaría estas mismas vistas a PROC GCHART o PROC SGPLOT. Pero como una primera pasada sin configuración sobre un extracto nuevo, estos gráficos responden las preguntas operativas —combinación, magnitud y momento— en segundos.